Encuentra la carpeta de datos desde el notebook

In [4]:
from pathlib import Path
import pandas as pd

actual = Path.cwd().resolve()
candidatas = [actual, *actual.parents]
ROOT = next(
    (ruta for ruta in candidatas if (ruta / "data" / "bronze").is_dir()),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "No encontré data/bronze. Abre el notebook desde el proyecto "
        "o ejecuta primero el pipeline."
    )

BRONZE = ROOT / "data" / "bronze"
archivos = sorted(BRONZE.glob("*.parquet"))

if not archivos:
    raise FileNotFoundError(
        f"No hay archivos Parquet en {BRONZE}. Ejecuta python -m src.pipeline."
    )

print(f"Proyecto: {ROOT}")
print(f"Parquet encontrados: {len(archivos)}")
for archivo in archivos:
    print("-", archivo.name)

Proyecto: /workspaces/observatorio-energia
Parquet encontrados: 11
- enerandes_ventas_2023.parquet
- enerandes_ventas_2024.parquet
- enerandes_ventas_2025.parquet
- enerandes_ventas_2026.parquet
- ideam_precipitacion_diaria.parquet
- ideam_temperatura_diaria.parquet
- noaa_oni.parquet
- trm_colombia.parquet
- xm_demanda.parquet
- xm_embalses.parquet
- xm_precio_bolsa.parquet


Ejecuta específicamente las líneas de fechas
En otra celda, carga IDEAM en df y luego calcula las fechas:

In [5]:
df = pd.read_parquet(BRONZE / "ideam_precipitacion_diaria.parquet")

fechas = pd.to_datetime(df["fecha"], errors="coerce")
print("Fecha inicial:", fechas.min())
print("Fecha final:", fechas.max())
print("Fechas que no se pudieron interpretar:", fechas.isna().sum())

Fecha inicial: 2025-01-01 00:00:00
Fecha final: 2025-03-11 00:00:00
Fechas que no se pudieron interpretar: 0


La conversión con errors="coerce" permite contar fechas inválidas en vez de detener la celda con un error. Si quieres probar XM, la columna se llama Date —con mayúscula—, no fecha.

Perfilamiento completo de los Parquet
Esta celda presenta filas, columnas, tipos, nulos, duplicados y rangos cuando encuentra columnas con fechas reconocibles:

In [ ]:
from IPython.display import display

columnas_fecha = {"date", "fecha", "vigenciadesde", "vigenciahasta", "yr"}

for archivo in archivos:
    df = pd.read_parquet(archivo)

    print(f"\n## {archivo.name}")
    print(f"Filas: {len(df):,} | Columnas: {len(df.columns)}")
    print(f"Filas duplicadas exactas: {df.duplicated().sum():,}")

    perfil_columnas = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "porcentaje_nulos": (df.isna().mean() * 100).round(2),
    })
    display(perfil_columnas)

    fechas_archivo = [
        columna for columna in df.columns
        if str(columna).lower() in columnas_fecha
    ]

    if fechas_archivo:
        for columna in fechas_archivo:
            fechas = pd.to_datetime(df[columna], errors="coerce")
            print(
                f"{columna}: {fechas.min()} a {fechas.max()} "
                f"| fechas inválidas: {fechas.isna().sum()}"
            )
    else:
        print("No se encontró una columna de fecha con nombre reconocido.")

En EnerAndes las columnas vienen como col_0, col_1, etc.; por eso este código no asume que col_0 sea siempre una fecha. En esos archivos también hay encabezados y títulos dentro de las filas, así que hay que interpretarlos antes de calcular un rango confiable.

Reproduce los hallazgos numéricos
Esta celda vuelve a calcular los hallazgos reportados, usando los archivos que encuentre localmente:

In [ ]:
# EnerAndes: duplicados y nulos en columnas de datos col_*
enerandes = sorted(BRONZE.glob("enerandes_ventas_*.parquet"))

if enerandes:
    filas_total = 0
    duplicados_total = 0
    nulos_total = 0

    for archivo in enerandes:
        df = pd.read_parquet(archivo)
        columnas_datos = [c for c in df.columns if c.startswith("col_")]

        filas_total += len(df)
        duplicados_total += int(df.duplicated().sum())
        nulos_total += int(df[columnas_datos].isna().sum().sum())

    print(
        f"EnerAndes: {duplicados_total:,} duplicados de {filas_total:,} filas "
        f"({duplicados_total / filas_total * 100:.2f} %)"
    )
    print(f"EnerAndes: {nulos_total:,} celdas nulas en columnas de datos")

    for archivo in enerandes:
        df = pd.read_parquet(archivo)
        columnas_datos = [c for c in df.columns if c.startswith("col_")]
        print(
            f"{archivo.name}: {len(df):,} filas, "
            f"{len(columnas_datos)} columnas de datos; "
            f"primeros valores de col_0: {df['col_0'].head(3).tolist()}"
        )

# IDEAM: cobertura para ambos datasets y conteos de observaciones
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src import config

dias_solicitados = pd.date_range(
    pd.Timestamp(config.IDEAM_FECHA_INICIO),
    pd.Timestamp(config.IDEAM_FECHA_FIN),
    freq="D",
)

for nombre in ("ideam_precipitacion_diaria", "ideam_temperatura_diaria"):
    archivo = BRONZE / f"{nombre}.parquet"
    if not archivo.exists():
        continue

    ideam = pd.read_parquet(archivo)
    fechas = pd.to_datetime(ideam["fecha"], errors="coerce").dropna().dt.normalize()
    dias_observados = pd.DatetimeIndex(fechas.unique())
    dias_faltantes = dias_solicitados.difference(dias_observados)
    print(
        f"{nombre}: {len(dias_observados)} fechas observadas; "
        f"{len(dias_faltantes)} días sin registros de "
        f"{len(dias_solicitados)} solicitados"
    )

    n_obs = pd.to_numeric(ideam["n_obs"], errors="coerce")
    sobre_144 = int((n_obs > 144).sum())
    print(
        f"{nombre}: {sobre_144:,} de {len(ideam):,} registros tienen "
        f"n_obs > 144; máximo={n_obs.max():,.0f}; "
        f"mediana={n_obs.median():,.0f}"
    )